# KNN from Scratch

Using the KNN algorithm from scikit-learn is quite simple. When you use it, you don't even need to know how KNN works and classifies the data points. The goal should be to really understand what you are doing. This is the only way to make sure you understand when something doesn't go the way you expect or want it to. Many people remember things best when they implement them themselves. So, let's build our own K-Nearest-Neighbors classifier!

**The purpose of this notebook is to help you remember the steps necessary to classify samples with KNN.**

To test if your code works, you can use the Iris dataset as a data example.
Let's make a plan and break this big task into smaller steps!


1. What information and data does the algorithm need to train and predict the classes of new instances?
This will be the input for our function! 

2. calculate the distance between the test point and each existing data point in the training data.
3. determine the nearest k neighbors.
4. make predictions based on these neighbors.

You have already implemented a function to calculate the distance between points, which will now come in handy.

A good way to get started, is to ignore the syntax and just write in simple text what you want your program to do aka **write pseudo-code**. You can then start to build out some of the structure. What variables are you going to need? What kinds of logic? 
Knowing where you’re going can help you make fewer mistakes as you’re trying to get there.

Note that for large data sets, the algorithm can take very long to classify because it has to calculate the distance between the test point and every other point in the data!

You can check if your pseudo-code contains all necessary steps afterwards, when scrolling down to "KNN algorithm from scratch" where you find an example of a knn pseudo-code.

## Import and Setup

In [1]:
import numpy as np
import pandas as pd
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

In [3]:
# Load data
df = pd.read_csv("data/iris.csv")
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


In [5]:
# Defining X and y

x = df.drop("species",  axis=1)
y = df["species"]

In [10]:
# Train test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

## Distance Metrics

As already explained, KNN assigns a class to the test point based on the majority class of  K  nearest neighbors. In general, euclidean distance is used to find nearest neighbors, but other distance metrics can also be used.

As the dimensionality of the feature space increases, the euclidean distance often becomes problematic due to the curse of dimensionality (discussed later).

In such cases, alternative vector-based similarity measures (dot product, cosine similarity, etc) are used to find the nearest neighbors. This transforms the original metric space into one more amenable to point-to-point measurements.

Another distance measure that you might consider is [Mahalanobis distance](https://en.wikipedia.org/wiki/Mahalanobis_distance). Mahalanobis distance attempts to weight features according to their probabilities. On some data sets that may be important.

In general, it's probably a good idea to normalize the data at a minimum. Here's a link to the scikit-learn scaling package: http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html . You have to be a little circumspect about employing any technique where the answers change with scaling.

In [8]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(120, 4)
(120,)
(30, 4)
(30,)


In [11]:
# Implemented own distance function
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)

print(confusion_matrix(y_test, y_pred))

[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


## KNN Algorithm from scratch


Remember the steps:

1. What information and data does the algorithm need to train and predict the classes of new instances?
This will be the input for our function! 

2. calculate the distance between the test point and each existing data point in the training data.
3. determine the nearest k neighbors.
4. make predictions based on these neighbors.

Hopefully you have already thought of your gameplan, also called pseudo-code. If so, you can compare it to this one:
```
INPUT: X_train, y_train, X_test, k
FOR each object_to_predict in X_test:
    FOR each training_point, index in X_train:
        calculate distance d between object_to_predict and training_point
        store d and index
    SORT distances d in increasing order
    take first k items, get indices of those
    calculate most common class of points at indices in y_train (prediction)
    store prediction
RETURN list of predictions
````

Time to code!
Don't forget that it's good practice to document your own code! This way you can later understand what the purpose of each step was.
Maybe you can even use your pseudo code as documentation :)

In [18]:
# Your code

# KNN from scratch

def knn(x_train, y_train, x_test, k=5):

    predictions = []  # store our predictions

    for test in x_test:  # go through each test point

        distances = []  # store distances to training points

        for i in range(len(x_train)):  # go through each training point

            # Calculate Euclidean distance
            diff = test - x_train[i]
            d = np.sqrt(np.sum(diff ** 2))

            # Store distance and the training point's label
            distances.append((d, y_train.iloc[i]))

        distances.sort()  # put nearest points first

        neighbors = distances[:k]  # take the 5 nearest neighbors

        labels = [label for d, label in neighbors]  # get their species

        # Choose the species that appears most often
        prediction = max(set(labels), key=labels.count)

        predictions.append(prediction)  # save the prediction

    return predictions  # return all predictions


# Run our KNN
y_pred = knn(
    x_train.to_numpy(),
    y_train,
    x_test.to_numpy()
)

# Check how well our predictions match the real labels
print(confusion_matrix(y_test, y_pred))

[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


## Comparison with sklearn knn implementation

That will be interesting! Check out how your implementation performs in comparison to the one of sklearn!
You can check the confusion matrix and the accuracy score of both algorithms.
If you want, you can check which algorithm is faster!

In [19]:
# Your code

# ==============================
# Compare our KNN with sklearn
# ==============================

# Run our own KNN implementation
y_pred_scratch = knn(
    x_train.to_numpy(),
    y_train,
    x_test.to_numpy()
)


# Create sklearn's KNN
knn_sklearn = KNeighborsClassifier(n_neighbors=5)

# Train sklearn's KNN
knn_sklearn.fit(x_train, y_train)

# Make predictions with sklearn's KNN
y_pred_sklearn = knn_sklearn.predict(x_test)


# ------------------------------
# Confusion matrices
# ------------------------------

print("My KNN:")
print(confusion_matrix(y_test, y_pred_scratch))

print("\nSklearn KNN:")
print(confusion_matrix(y_test, y_pred_sklearn))


# ------------------------------
# Accuracy
# ------------------------------

# Number of correct predictions / total predictions
accuracy_scratch = np.mean(
    np.array(y_pred_scratch) == y_test.to_numpy()
)

accuracy_sklearn = np.mean(
    y_pred_sklearn == y_test.to_numpy()
)

print("\nMy KNN accuracy:", accuracy_scratch)
print("Sklearn KNN accuracy:", accuracy_sklearn)

My KNN:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Sklearn KNN:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

My KNN accuracy: 1.0
Sklearn KNN accuracy: 1.0
